# Exercise 3.4 — Gradient descent (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day3/day3-03-gradient-descent.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

This is the longest exercise of the day and the only one where you will watch an
algorithm stop far from the answer without any warning. Run the cells as you read; the printed output is what you should
get.

## Objectives

- Write the gradient descent update rule and say what each symbol in it does.
- Minimise a function in code, and tell from the printed path whether the
  learning rate is too large, too small, or about right.
- Explain why gradient descent on unscaled data can stop far from the answer
  without any warning, and how standardising fixes it.

## Why this, here

Exercise 3.3 found the lowest point of a loss surface by evaluating
15,251 candidate lines. That works for two parameters and nothing larger. Larger
models — logistic regression, and anything with a neural network in it — are
fitted by the method on this page instead: start somewhere, measure which way is downhill, take a step, repeat.

### The update rule

$$x_{n+1} = x_n - \alpha f'(x_n)$$

- $x_n$ is the current guess.
- $f'(x_n)$ is the derivative of the function at that guess: the slope of the
  tangent to the curve there. When we fit a model, the function is the MSE and
  $x$ is one of the coefficients, so this slope is **the derivative of the MSE
  with respect to the coefficient**.
- $\alpha$ is the learning rate: a small positive number controlling step size.

The minus sign carries the whole idea. Where the slope is positive the function
rises to the right, so we subtract and move left; where it is negative we
subtract a negative and move right. Either way the step is *against* the slope,
and the steeper the slope the bigger the step.

### Minimising $f(x) = x^2$

The derivative is $f'(x) = 2x$, so the rule becomes $x_{n+1} = x_n - 2\alpha
x_n$. We know the answer is $x = 0$, which is why this is the example: you can
see whether the algorithm is working.

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import numpy as np
import matplotlib.pyplot as plt

f       = lambda x: x**2
f_prime = lambda x: 2 * x

xs = np.linspace(-10, 10, 200)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(xs, f(xs), color="steelblue");  axes[0].set_title("f(x) = x^2")
axes[1].plot(xs, f_prime(xs), color="firebrick"); axes[1].set_title("f'(x) = 2x")
for ax in axes:
    ax.axhline(0, color="black", lw=0.8); ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("x")
plt.tight_layout(); plt.show()

Start at $x = -10$ with $\alpha = 0.8$ and take ten steps.

In [ ]:
alpha = 0.8
x     = -10.0
path  = []

for i in range(1, 11):
    x = x - alpha * f_prime(x)
    path.append(x)
    print(f"step {i}: x = {x:.4g}  f(x) = {f(x):.4g}")

The guess goes −10, 6, −3.6, 2.16, −1.296 and so on: it crosses the minimum
every single step and lands closer each time. Overshooting is not failure. With
$\alpha = 0.8$ each step is 1.6 times the distance to the minimum, so the sign
flips and the magnitude shrinks by 0.6.

In [ ]:
fig, ax = plt.subplots()
ax.plot(xs, f(xs), color="grey")
sc = ax.scatter(path, [f(v) for v in path], c=range(1, len(path) + 1),
                cmap="autumn_r", edgecolor="black", zorder=3)
cbar = fig.colorbar(sc, ax=ax, label="step")
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
plt.show()

### Task 1 — Your turn, on a function whose minimum you can check

Minimise

$$f(x) = x^2 + 3x + 5$$

Differentiate it on paper first: you should get $f'(x) = 2x + 3$, which is zero
at $x = -1.5$. Then write the loop and see whether the algorithm agrees with
your algebra.

> **Worked answer**

In [ ]:
g       = lambda x: x**2 + 3 * x + 5
g_prime = lambda x: 2 * x + 3

alpha = 0.1
x = 2.0
for i in range(40):
    x = x - alpha * g_prime(x)

print(round(x, 4))   # your algebra said -1.5

Forty steps get you to −1.4995. The algorithm approaches the minimum and never
arrives, which is true of gradient descent in general: you stop when the
improvement stops mattering, not when the gradient is zero.

### The learning rate is the whole game

Run the same problem — $f(x) = x^2$ from $x = -10$ — at four learning rates for
twenty steps each.

In [ ]:
import pandas as pd

def descend_1d(alpha, x0=-10.0, n=20):
    x = x0
    for _ in range(n):
        x = x - alpha * f_prime(x)
    return x

rates = [0.01, 0.1, 0.8, 1.0, 1.05]
pd.DataFrame({"alpha": rates,
              "final_x": [descend_1d(a) for a in rates],
              "final_f": [f(descend_1d(a)) for a in rates]})

Read that table carefully, because every row is a different lesson.

| $\alpha$ | After 20 steps | What happened |
|---|---|---|
| 0.01 | $x = -6.68$ | correct direction, far too slow — it needs thousands of steps |
| 0.1 | $x = -0.115$ | steady approach from one side |
| 0.8 | $x = -0.00037$ | overshoots every step and converges fastest of the five |
| 1.0 | $x = -10$ | bounces between −10 and 10 for ever, making no progress at all |
| 1.05 | $x = -67.3$ | diverges: each step is further out than the last |

The fastest rate here is not the smallest one, and the boundary between working
and diverging is sharp. For $f(x)=x^2$ the step multiplies the distance to the
minimum by $(1 - 2\alpha)$, so anything below 1 converges, 1 exactly cycles, and
above 1 explodes. There is no warning and no error message: at $\alpha = 1.05$
the loop runs happily and returns a number.

> **A run that finished is not a run that worked**
>
> Nothing in any of those five runs raised an error. If you only look at the final
> number, the diverging run and the converging run are indistinguishable in form —
> both are a number. Always print the path, or the loss, and look at whether it is
> going down.

> **One dip or several**
>
> Some functions have more than one dip, and gradient descent stops in whichever
> one it walks into first: start somewhere else and you can get a different
> answer, with no warning. The MSE of a linear regression is a single bowl, as you
> saw in Exercise 3.3, so every starting point reaches the same line. Most other
> models are not so kind, which is why the same neural network trained twice can
> give two different results.

### Gradient descent on the glucose data

Now put the method on the actual problem: find the intercept and slope that
minimise the MSE for glucose against age, by gradient descent instead of by
`LinearRegression`. The derivatives of
$\text{MSE}(\beta_0, \beta_1) = \frac{1}{m}\sum (y_i - \beta_0 - \beta_1 x_i)^2$
with respect to the two coefficients are

$$\frac{\partial \text{MSE}}{\partial \beta_0} = -\frac{2}{m}\sum r_i
\qquad
\frac{\partial \text{MSE}}{\partial \beta_1} = -\frac{2}{m}\sum x_i r_i$$

with $r_i = y_i - \beta_0 - \beta_1 x_i$ (the extra reading at the end of Day 3 derives
them). Two coefficients means two derivatives, and the update rule is applied to
both at the same time.

In [ ]:
import pandas as pd
glu = pd.read_csv("data/glucose_study.csv")

def descend_mse(x, y, alpha, n):
    x, y = np.asarray(x, float), np.asarray(y, float)
    b0 = b1 = 0.0
    mse = np.empty(n)
    for i in range(n):
        r = y - (b0 + b1 * x)
        b0 = b0 - alpha * (-2 * r.mean())
        b1 = b1 - alpha * (-2 * (x * r).mean())
        mse[i] = np.mean((y - (b0 + b1 * x))**2)
    return {"b0": b0, "b1": b1, "mse": mse}

raw = descend_mse(glu["age"], glu["glucose"], alpha=1e-4, n=1000)
raw["b0"], raw["b1"], raw["mse"][-1]

After a thousand iterations the descent reports an intercept of 3.23 and a slope
of 3.04, with an MSE of 2513. `LinearRegression` gives 110.30 and 0.289, with an MSE of 847.5.

**Why is it so far off? The descent has not finished.** Both coefficients start
at 0. The intercept has to travel all the way to about 110, the slope only to
about 0.3. But the derivative of the MSE with respect to the slope is multiplied
by age (33 years on average), so at the start it is about 34 times larger than
the derivative with respect to the intercept. The learning rate has to be small
enough that the slope does not overshoot, and at that rate the intercept
crawls: it needs about 160,000 iterations to get within 1 of 110. Make the
learning rate ten times larger ($10^{-3}$) and the slope overshoots further at
every step until the numbers blow up.

Nothing warned us. The MSE went down at every iteration and the loop finished
without an error. Compare with a known answer when you have one, and always
look at the loss.

The fix is to standardise both columns (subtract the mean, divide by the
standard deviation) so that both coefficients are on the same scale, descend
there, and convert the answer back.

### Task 2 — Make it converge

Standardise, descend with a learning rate a thousand times larger, and rescale
the parameters to the original units.

> **Worked answer**

In [ ]:
from sklearn.linear_model import LinearRegression

mx, sx = glu["age"].mean(), glu["age"].std()
my, sy = glu["glucose"].mean(), glu["glucose"].std()
zx = (glu["age"] - mx) / sx
zy = (glu["glucose"] - my) / sy

std = descend_mse(zx, zy, alpha=0.1, n=50)

slope     = std["b1"] * sy / sx
intercept = my + sy * std["b0"] - slope * mx

ref = LinearRegression().fit(glu[["age"]], glu["glucose"])
print(intercept, slope)
print(round(ref.intercept_, 3), round(ref.coef_[0], 4))   # the answer to compare with

Fifty iterations on the standardised data give 110.298 and 0.28867, agreeing
with `LinearRegression` to five significant figures. The same algorithm, the same data, the
same loss; twenty times fewer iterations and a thousand times larger step.
Scaling was not a tidying-up step, it was the difference between an answer and
no answer.

In [ ]:
std_shown = descend_mse((glu["age"] - glu["age"].mean()) / glu["age"].std(),
                        (glu["glucose"] - glu["glucose"].mean()) / glu["glucose"].std(),
                        alpha=0.1, n=50)

fig, ax = plt.subplots()
ax.plot(range(1, 51), std_shown["mse"], color="firebrick")
ax.set_xlabel("Iteration"); ax.set_ylabel("MSE (standardised units)")
plt.show()

That curve is the plot to draw every time you run an optimiser. A descent that
is working looks like this: a steep fall, then a flat tail. Anything else — a
rise, a wobble, a straight line still sloping down at the last iteration — is
telling you to change the learning rate or the number of steps.

## Reflection

1. You are handed a fitted model and a plot of its loss against iteration. The
   curve is still falling at the final iteration. What do you do, and what would
   you have to check before trusting the model as it stands?
2. Standardising changed the parameters you had to find but not the model. Why
   can the two intercepts — one on standardised columns, one on the original —
   differ while the predictions stay identical?

> **Learning points**
>
> - Gradient descent repeats one step: move each coefficient against the
>   derivative of the MSE with respect to that coefficient, scaled by the learning
>   rate.
> - The learning rate decides everything: too small is slow, too large overshoots
>   and blows up, and neither raises an error.
> - On unscaled columns the descent can stop far from the answer; standardise
>   first, and always look at the loss curve.
> - For linear regression the MSE is a single bowl, so the descent reaches the
>   same line as `lm()` and `LinearRegression`.